# Template application

In [ ]:
entity_template_file = 'entity.templ.html'
attribute_template_file = 'attribute.templ.html'
system_template_file = 'system.templ.html'

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>Invalid information. For testing purposes only!</b></span> ')

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
from lxml import etree
import IPython

# valid xml namespaces and schema for Confluence 6 storage format
# See 
xml_namespaces=[
    'xmlns="http://www.w3.org/1999/xhtml"',
    'xmlns:ac="http://www.atlassian.com/schema/confluence/4/ac/"',
    'xmlns:ri="http://www.atlassian.com/schema/confluence/4/ri/"',
    'xmlns:acxhtml="http://www.atlassian.com/schema/confluence/4/"'
]

def encapsulate_storage_format(xml):
    return '<?xml version="1.0"?><root doc="container to properly encapsulate xml" {} >\n{}\n</root>'.format(
            ' '.join(xml_namespaces), xml)

def beautify_xml(flat_xml):
    try:
        encapsulated = encapsulate_storage_format(flat_xml)
        dom = xml.dom.minidom.parseString(encapsulated)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        log.error('Expat error {}'.format(e))
        raise Exception(e)
    
def validate_storage_format(xml_in_storage_format):
        '''Validate if input xml conforms to Confluence storage format specifications'''
        parser = etree.XMLParser(dtd_validation=False)
        try:
            etree.fromstring(encapsulate_storage_format(xml_in_storage_format), parser)
            return None
        except xml.parsers.expat.ExpatError as e:
            m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
            message = 'Malformed xml ' + flat_xml[:50] + ' ...'
            if m:
                lines = wrapped.splitlines()
                messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
            else:
                message = message + flat_xml[:50] + ' ...'
            return message

beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('all fine!')

## Import the Publisher from Publisher.py
PublisherDev is an extension of the external Publisher in Publisher.py. Stable extensions to Publisher should be moved to Publisher.py.

In [ ]:
import Publisher as cp

class PublisherDev(cp.Publisher):
    '''This class'''
    
    def enable_development(self):
        self.version_comment = 'Update due to development / testing' 


# Load datasource

In [ ]:
import json

data = None
with open('testdata/IM_CRM_FYAYC.json', 'r') as source:
     data = json.load(source)

entities = data['entities']
# Print some information on what was loaded
print('Model "{}" contains {} entities:'.format(data['model']['name'], len(entities)))

In [ ]:
config = { 'content':
            {
                'entities': { 
                    'content': {
                      'attributes': {}
                    }  
                },
                'documents': {},
                'domains': {},
                'systems': {
                    'content': {
                        'tables': {
                            'content': {
                                'columns': {},
                            }
                        },
                    }
                },
            } 
         }
publisher = PublisherDev(config, data, None, None, None, languages=['de', 'en', 'fr'])
publisher.enable_development()
list(map(lambda e: (e, publisher.translate(entities[e]['name'])), data['entities']))[:5]

## Load page mappings
This will be done during the scan phase in production mode.

In [ ]:
topics = publisher.collect_recursive(publisher.config)
topics

In [ ]:
content = publisher.scan_current_content()
len(content)

In [ ]:
column_key = list(data['columns'])[0]
column = data['columns'][column_key]

attribute_key = column['attributes-mapped'][0]
attribute_key

In [ ]:
attribute = data['attributes'][attribute_key]
attribute

In [ ]:
columns_mapped = attribute['columnsmapped+']
columns_mapped

In [ ]:
other_columns = map(lambda entry: columns_mapped[entry], columns_mapped) 
col_list = list(other_columns)
col_list

In [ ]:
from functools import reduce
cols = reduce(lambda e, l: e + l, col_list)
cols

In [ ]:
lineage = publisher.column_lineage(column_key)
'Lineage of {} = {}'.format(column_key, lineage)

In [ ]:
publisher.attribute_lineage(attribute_key)

In [ ]:
column_key

In [ ]:
elements = [ 'entities', 'attributes', 'relations', 'tables', 'domains', 'columns']
# Fake page id's
rnd = -1
for element_type in elements:
    for key in data[element_type]:
        entitiy = data[element_type][key]
        name = entitiy.get('name')
        rnd -= 1
        publisher.register_page_id(key, rnd)

In [ ]:
relations = list(filter(lambda rel: data['relations'][rel], data['relations']))

pick = data['relations'][relations[0]]
pick

In [ ]:
one_end = pick['from-to']['enti']
lhs = publisher.relation_other(one_end, relations[0])
lhs

In [ ]:
rhs = publisher.relation_self(one_end, relations[0])
rhs

In [ ]:
assert one_end == rhs['enti']
assert rhs != lhs

# Prepare to test samples

In [ ]:
import os
output_folder = 'target'
os.makedirs(output_folder, exist_ok=True)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
env.globals.update({ 'util': publisher, 'data': data, 'header': header })

# Entity template sandbox

In [ ]:
test_entity_key = list(filter(lambda e: len(data['entities'][e]['attributes+']) > 2, data['entities']))[0]
assert test_entity_key
test_entity = data['entities'][test_entity_key]

entity_template = env.get_template(entity_template_file)
rendered_entity_template = entity_template.render(key=test_entity_key, item=test_entity)
with open(os.path.join(output_folder, 'entity.out'), 'w') as out:
    out.write(rendered_entity_template)

In [ ]:
entity_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_entity_template) # strip comment lines
IPython.display.Code(str(entity_content_xml)[:512])

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(publisher.page_title(test_entity_key), entity_content_xml))

In [ ]:
validate_storage_format(entity_content_xml)

# Sandbox for attribute template

In [ ]:
first_attribute_key = list(data['attributes'])[0]
assert first_attribute_key
first_attribute = data['attributes'][first_attribute_key]

test_attribute_title = publisher.page_title(first_attribute_key)
log.warning('Working with attribute ' + first_attribute_key + ". Name: " + test_attribute_title)
first_attribute

In [ ]:
attribute_template = env.get_template(attribute_template_file)
rendered_attribute_template = attribute_template.render(key=first_attribute_key, item=first_attribute, util=publisher, header=header)
attribute_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_attribute_template) # strip comment lines
#IPython.display.Code(attribute_content_xml)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(test_attribute_title, attribute_content_xml))

# System template sandbox

In [ ]:
system_template = env.get_template(system_template_file)
first_system_key = list(data['systems'])[0]
first_system = data['systems'][first_system_key]
rendered_system_template = system_template.render(key=first_system_key, item=first_system, util=publisher, header=header)

with open(os.path.join(output_folder, 'system.out'), 'w') as output:
    output.write(rendered_system_template)

In [ ]:
attribute_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_system_template) # strip comment lines
#IPython.display.Code(attribute_content_xml)

In [ ]:
domain_template = env.get_template('domain.templ.html')
first_domain_key = list(data['domains'])[0]
first_domain = data['domains'][first_domain_key]
rendered_system_template = domain_template.render(key=first_domain_key, item=first_domain, util=publisher, header=header)

with open(os.path.join(output_folder, 'domain.out'), 'w') as output:
    output.write(rendered_system_template)

In [ ]:
document_template = env.get_template('document.templ.html')

for document_key in list(data['documents']):
    first_document = data['documents'][document_key]
    rendered_system_template = document_template.render(key=document_key, item=first_document, util=publisher, header=header)

    with open(os.path.join(output_folder, 'document-' + document_key + '.out'), 'w') as output:
        output.write(rendered_system_template)

# Publish to test space to verify result in Confluence

In [ ]:
import yaml
import copy

with open('private.yaml') as f:
    config = yaml.safe_load(f)

space_key = config['confluence']['space']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])

In [ ]:
#entity_result = confluence.update_or_create(root_page_id, '{} - Test entity'.format(test_entity_title), entity_content_xml, minor_edit=False, version_comment='development and testing')
#attribute_result = confluence.update_or_create(root_page_id, '{} - Test attribute'.format(test_attribute_title), attribute_content_xml, minor_edit=True, version_comment='development and testing')

In [ ]:
# Need a real publisher
real_publisher = cp.Publisher(config, data, confluence, space_key, root_page_id)
complete = real_publisher.scan_current_content()
len(complete)

In [ ]:
elements = [ 'entities', 'attributes', 'relations', 'tables', 'domains', 'columns', 'systems']
# Fake page id's
rnd = -1
for element_type in elements:
    for key in data[element_type]:
        entitiy = data[element_type][key]
        name = entitiy.get('name')
        rnd -= 1
        real_publisher.register_page_id(key, rnd)
        page = real_publisher.page_for_key(key)
        page['title'] = key

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

real_env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
real_env.globals.update({ 'util': real_publisher, 'data': data, 'header': header, 'config': config })

In [ ]:
table_template = real_env.get_template('table.templ.html')

for table_key in list(data['tables'])[:3]:
    table = data['tables'][table_key]
    print('Rendering table ' + table_key)
    rendered_system_template = table_template.render(key=table_key, item=table)

    with open(os.path.join(output_folder, 'table-' + table_key + '.out'), 'w') as output:
        output.write(rendered_system_template)

In [ ]:
table_template = real_env.get_template('system.templ.html')

for table_key in list(data['systems'])[:3]:
    table = data['systems'][table_key]
    print('Rendering system ' + table_key)
    rendered_system_template = table_template.render(key=table_key, item=table)

    with open(os.path.join(output_folder, 'system-' + table_key + '.out'), 'w') as output:
        output.write(rendered_system_template)

In [ ]:
column_template = real_env.get_template('column.templ.html')

for column_key in list(data['columns'])[:3]:
    column = data['columns'][column_key]
    print('Rendering column ' + column_key)
    rendered_column_template = column_template.render(key=column_key, item=column)

    with open(os.path.join(output_folder, 'column-' + table_key + '.out'), 'w') as output:
        output.write(rendered_column_template)

In [ ]:
table_template = real_env.get_template('table.templ.html')

for table_key in list(data['tables'])[:3]:
    table = data['tables'][table_key]
    print('Rendering table ' + table_key)
    rendered_system_template = table_template.render(key=table_key, item=table)

    with open(os.path.join(output_folder, 'table-' + table_key + '.out'), 'w') as output:
        output.write(rendered_system_template)

In [ ]:
e_page = entity_result['_links']['webui']
e_uri = '{}{}'.format(config['confluence']['apiurl'], e_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    e_uri, test_entity_title))

In [ ]:
a_page = attribute_result['_links']['webui']
a_uri = '{}{}'.format(config['confluence']['apiurl'], a_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    a_uri, test_attribute_title))